# AI Energy Consumption Optimizer
## Phase 4: Model Training & Selection

This notebook trains and evaluates a curated selection of machine learning models for time-series forecasting.

### Model Selection Strategy & Justification
We evaluate exactly one representative from each major model family to structurally prove which mathematical approach best fits our energy load profile:
1. **Naive Baseline (T-168):** The absolute minimum bar. Energy consumption is highly cyclical; if a complex ML model cannot beat the simple logic of \"predict exactly what happened this hour last week\", the ML model introduces unnecessary risk.
2. **Linear Regression (Linear Family):** Tests if the relationships between our lag features and the target are purely linear. Energy consumption typically has non-linear thresholds (e.g., sudden HVAC spikes), so this serves as our ML baseline.
3. **Random Forest (Bagging Tree Family):** Tests if bagging multiple independent decision trees captures the non-linear boundaries better than a linear model without overfitting.
4. **XGBoost & LightGBM (Boosting Family):** These are our mathematically primary candidates. Gradient Boosting is exceptionally suited for time-series tabular data with sudden spikes because trees are built sequentially to correct the residual errors of previous trees, perfectly capturing anomalous peak-load behaviors.

In [ ]:
from google.colab import drive
import os
import pandas as pd
import numpy as np
import joblib
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

# Mount Drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/energy-optimizer'
print(f"Storage Bound: {BASE}")

Mounted at /content/drive
Storage Bound: /content/drive/MyDrive/energy-optimizer


In [ ]:
# 1. Load Processed, UNscaled Splits (Strict Temporal Order)
train_df = pd.read_csv(os.path.join(BASE, 'data/splits/train_unscaled.csv'), index_col='Datetime', parse_dates=True)
val_df   = pd.read_csv(os.path.join(BASE, 'data/splits/val_unscaled.csv'),   index_col='Datetime', parse_dates=True)
test_df  = pd.read_csv(os.path.join(BASE, 'data/splits/test_unscaled.csv'),  index_col='Datetime', parse_dates=True)

# Load feature map guaranteeing identical API ingestion
with open(os.path.join(BASE, 'artifacts/scalers/feature_columns.json')) as f:
    feature_cols = json.load(f)

TARGET = 'consumption'

X_train_raw, y_train = train_df[feature_cols], train_df[TARGET]
X_val_raw, y_val     = val_df[feature_cols], val_df[TARGET]
X_test_raw, y_test   = test_df[feature_cols], test_df[TARGET]

# Fit scaler ONLY on training data with the full final feature set
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

# Critical schema guard: scaler dimension must match feature schema size
assert scaler.n_features_in_ == len(feature_cols), (
    f"Mismatch: scaler expects {scaler.n_features_in_}, feature list has {len(feature_cols)}"
)

print(f"Temporal Splits Confirmed -> Train: {X_train_raw.shape}, Val: {X_val_raw.shape}, Test: {X_test_raw.shape}")
print(f"Schema Check -> scaler.n_features_in_: {scaler.n_features_in_}, feature columns: {len(feature_cols)}")

Temporal Splits Confirmed -> Train: (23923, 16), Val: (5127, 16), Test: (5127, 16)


---
### Step 1: The Naive Baseline (T-168)
This establishes our minimum predictive performance threshold.

In [ ]:
# The 'lag_168' feature flawlessly represents exactly what the consumption was one week ago.
naive_preds = val_df['lag_168'].values

naive_rmse = np.sqrt(mean_squared_error(y_val, naive_preds))
naive_mae  = mean_absolute_error(y_val, naive_preds)

print("=== NAIVE BASELINE (Same Hour, Last Week) ===")
print(f"Baseline RMSE: {naive_rmse:.4f}")
print(f"Baseline MAE:  {naive_mae:.4f}")
print(f"\nRule: Any ML model failing to beat an RMSE of {naive_rmse:.4f} will be rejected.")

=== NAIVE BASELINE (Same Hour, Last Week) ===
Baseline RMSE: 0.9794
Baseline MAE:  0.6738

Rule: Any ML model failing to beat an RMSE of 0.9794 will be rejected.


---
### Step 2: Evaluating the Representative Model Families
We fit each model on the global training split and evaluate securely on the holdout validation split.

In [ ]:
def evaluate_model(model, name):
    """Fits model, predicts validation, logs metrics."""
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae  = mean_absolute_error(y_val, preds)
    mape = mean_absolute_percentage_error(y_val, preds) * 100
    return {'Model': name, 'Val_RMSE': round(rmse, 4), 'Val_MAE': round(mae, 4), 'Val_MAPE%': round(mape, 2)}

results = [{'Model': 'Naive Baseline (T-168)', 'Val_RMSE': round(naive_rmse, 4), 'Val_MAE': round(naive_mae, 4), 'Val_MAPE%': '-'}]

# 1. Linear Family
lr = LinearRegression()
results.append(evaluate_model(lr, 'Linear Regression'))

# 2. Bagging Tree Family
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
results.append(evaluate_model(rf, 'Random Forest'))

# 3. Boosting Family A (XGBoost)
xgb_model = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, tree_method='hist')
results.append(evaluate_model(xgb_model, 'XGBoost (Primary Candidate)'))

# 4. Boosting Family B (LightGBM)
lgbm_model = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42, verbose=-1)
results.append(evaluate_model(lgbm_model, 'LightGBM'))

Training Linear Regression...
Training Random Forest...
Training XGBoost (Primary Candidate)...
Training LightGBM...


---
### Step 3: Model Selection & Justification
We mathematically rank the models by their RMSE on the validation set, select the absolute winner, and provide the selection rationale.

In [ ]:
comparison_df = pd.DataFrame(results).sort_values('Val_RMSE')

print("\n" + "="*60)
print("         MODEL COMPARISON TABLE (Validation Set)")
print("="*60)
print(comparison_df.to_string(index=False))
print("="*60)

# Assuming Boosting models will win due to problem architecture, we dynamically select the #1 rank.
# We filter out the baseline just in case to ensure a real ML model is captured.
ml_models_only = comparison_df[comparison_df['Model'] != 'Naive Baseline (T-168)']
winner_row = ml_models_only.iloc[0]
winner_name = winner_row['Model']

print(f"\n🏆 SELECTION JUSTIFICATION:")
print(f"{winner_name} is automatically selected for production because it achieved the lowest RMSE ({winner_row['Val_RMSE']}) on the held-out validation set. Its gradient boosting architecture optimally captures the non-linear volatility of energy spikes better than linear algorithms.")


         MODEL COMPARISON TABLE (Validation Set)
                      Model  Val_RMSE  Val_MAE Val_MAPE%
                   LightGBM    0.5792   0.4020    470.49
XGBoost (Primary Candidate)    0.5798   0.4022    463.71
              Random Forest    0.5873   0.4043    457.92
          Linear Regression    0.6359   0.4564    245.73
     Naive Baseline (T-168)    0.9794   0.6738         -

🏆 SELECTION JUSTIFICATION:
LightGBM is automatically selected for production because it achieved the lowest RMSE (0.5792) on the held-out validation set. Its gradient boosting architecture optimally captures the non-linear volatility of energy spikes better than linear algorithms.


---
### Step 4: Final Test Set Evaluation (One-Shot)
To ensure complete integrity, the winning model touches the Test Set exactly once to compute our final production metrics.

In [ ]:
# Map winner name to actual model object
model_map = {'Linear Regression': lr, 'Random Forest': rf, 'XGBoost (Primary Candidate)': xgb_model, 'LightGBM': lgbm_model}
final_model = model_map[winner_name]

test_preds = final_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae  = mean_absolute_error(y_test, test_preds)
test_mape = mean_absolute_percentage_error(y_test, test_preds) * 100

print(f"\nFINAL PRODUCTION METRICS (Test Set, One-Shot)")
print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")


FINAL PRODUCTION METRICS (Test Set, One-Shot)
Test RMSE: 0.5035
Test MAE:  0.3483


---
### Step 5: Uncertainty Quantification (Prediction Intervals)
We natively train q05 and q95 models using XGBoost's quantile loss objective. This allows the API to return a 90% confidence band alongside the single-point prediction.

In [ ]:
print("Training Upper and Lower Bound Interval Models (XGBoost Quantile)...")

xgb_q05 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.05, n_estimators=300, max_depth=6, learning_rate=0.05, tree_method='hist', random_state=42)
xgb_q05.fit(X_train, y_train)

xgb_q95 = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.95, n_estimators=300, max_depth=6, learning_rate=0.05, tree_method='hist', random_state=42)
xgb_q95.fit(X_train, y_train)

print("✅ Quantile Models Built.")

Training Upper and Lower Bound Interval Models (XGBoost Quantile)...
✅ Quantile Models Built.


---
### Step 6: Artifact Management & Metadata Registry
We save **ONLY** the absolute winner and its interval models explicitly into the API's `metadata.json` simulation linker.

In [ ]:
# File definitions
model_file = "forecast_winner.pkl"
q05_file = "xgb_q05.pkl"
q95_file = "xgb_q95.pkl"
scaler_file = "main_scaler.pkl"
feature_file = "feature_columns.json"

# 1. Save Physical Models to /models/
joblib.dump(final_model, os.path.join(BASE, f'artifacts/models/{model_file}'))
joblib.dump(xgb_q05, os.path.join(BASE, f'artifacts/models/{q05_file}'))
joblib.dump(xgb_q95, os.path.join(BASE, f'artifacts/models/{q95_file}'))

# 2. Save Scaler + Feature Schema together (critical for inference consistency)
joblib.dump(scaler, os.path.join(BASE, f'artifacts/scalers/{scaler_file}'))
with open(os.path.join(BASE, f'artifacts/scalers/{feature_file}'), 'w') as f:
    json.dump(feature_cols, f)

# 3. Hard validation guard before registry write
assert scaler.n_features_in_ == len(feature_cols), (
    f"Mismatch before save: scaler={scaler.n_features_in_}, features={len(feature_cols)}"
)

# 4. Create the Metadata JSON Registry for FastAPI
metadata = {
    "version_tag": "Production-v1",
    "algorithm": winner_name,
    "model_filename": model_file,
    "q05_filename": q05_file,
    "q95_filename": q95_file,
    "scaler_filename": scaler_file,
    "feature_columns": feature_file,
    "metrics": {
        "test_rmse": round(test_rmse, 4),
        "test_mae": round(test_mae, 4)
    }
}

with open(os.path.join(BASE, 'artifacts/metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=4)

print("\n[OK] ARTIFACTS SAVED TO GOOGLE DRIVE")
print(f"[OK] scaler.n_features_in_: {scaler.n_features_in_}")
print(f"[OK] len(feature_cols): {len(feature_cols)}")
print(json.dumps(metadata, indent=4))
print("\nPhase 4 is complete with aligned model/scaler/feature schema.")


⭐ ARTIFACTS DUMPED SAFELY TO GOOGLE DRIVE ⭐
{
    "version_tag": "Production-v1",
    "algorithm": "LightGBM",
    "model_filename": "forecast_winner.pkl",
    "q05_filename": "xgb_q05.pkl",
    "q95_filename": "xgb_q95.pkl",
    "scaler_filename": "main_scaler.pkl",
    "feature_columns": "feature_columns.json",
    "metrics": {
        "test_rmse": 0.5035,
        "test_mae": 0.3483
    }
}

Phase 4 is complete. You may download models and the updated metadata.json to your local folder to run the backend.
